# CLIP and BERT Embeddings: Research and Methodology

## Objective
This notebook provides comprehensive understanding of:
1. **CLIP** (Contrastive Language-Image Pre-training) for visual feature extraction
2. **BERT** (Bidirectional Encoder Representations from Transformers) for textual feature extraction
3. **Multi-modal integration** for influencer-brand mapping

## Table of Contents
1. Introduction to Embeddings
2. CLIP Architecture and Applications
3. BERT Architecture and Applications
4. Why CLIP and BERT for Influencer-Brand Mapping
5. Feature Extraction Pipeline
6. Pre-trained vs. Fine-tuned Models
7. Research Methodology

## 1. Introduction to Embeddings

### 1.1 What are Embeddings?

**Definition**: Embeddings are dense, fixed-length vector representations that capture semantic meaning of data (text, images, etc.) in a continuous space.

**Key Properties**:
- **Dense**: Every dimension contains information (vs. sparse one-hot encoding)
- **Fixed-length**: Consistent dimensionality regardless of input size
- **Semantic**: Similar items have similar vectors (measured by cosine similarity or Euclidean distance)

### 1.2 Why Use Embeddings?

**Traditional Approaches (Problems)**:
- Bag-of-words: Loses word order and context
- TF-IDF: High dimensionality, no semantic understanding
- Manual features: Labor-intensive, domain-specific

**Embedding Advantages**:
- Capture semantic relationships
- Enable transfer learning (pre-trained models)
- Work across domains and tasks
- Enable efficient similarity search

### 1.3 Embedding Space

```
High-dimensional space where:
- Similar concepts cluster together
- Semantic relationships preserved
- Arithmetic operations have meaning

Example:
  vector("fitness influencer") + vector("product") 
  ≈ vector("sponsored fitness product")
```

**Similarity Metrics**:
```python
# Cosine Similarity (most common)
similarity = (A · B) / (||A|| × ||B||)
# Range: [-1, 1], 1 = identical, -1 = opposite

# Euclidean Distance
distance = sqrt(sum((A - B)²))
# Range: [0, ∞), 0 = identical
```

## 2. CLIP: Contrastive Language-Image Pre-training

### 2.1 What is CLIP?

**Developed by**: OpenAI (2021)

**Core Idea**: Learn visual concepts from natural language supervision by training on 400 million (image, text) pairs from the internet.

**Key Innovation**: Joint embedding space for images AND text
- Images and their descriptions are mapped to nearby points
- Enables zero-shot classification
- Cross-modal retrieval (find images by text, text by images)

### 2.2 CLIP Architecture

```
Input Image → Image Encoder (Vision Transformer) → Image Embedding (512-dim)
                                                          ↓
                                                    Cosine Similarity
                                                          ↑
Input Text  → Text Encoder (Transformer) → Text Embedding (512-dim)
```

**Components**:
1. **Image Encoder**: Vision Transformer (ViT) or ResNet
   - Converts image to 512-dimensional vector
   - Captures visual concepts, objects, scenes, styles

2. **Text Encoder**: Transformer (similar to GPT)
   - Converts text to 512-dimensional vector
   - Understands natural language descriptions

3. **Contrastive Learning**:
   - Maximize similarity for matching (image, text) pairs
   - Minimize similarity for non-matching pairs

### 2.3 CLIP Capabilities

**Zero-Shot Classification**:
```python
# Without any training, classify image into categories
image_embedding = clip.encode_image(image)
text_embeddings = clip.encode_text(["a yoga mat", "a dumbbell", "a protein shake"])
similarities = cosine_similarity(image_embedding, text_embeddings)
predicted_class = categories[similarities.argmax()]
```

**Image Similarity**:
```python
# Find similar product images
thumbnail1 = clip.encode_image(img1)  # Fitness video thumbnail
thumbnail2 = clip.encode_image(img2)  # Another fitness thumbnail
similarity = cosine_similarity(thumbnail1, thumbnail2)
```

**Text-to-Image Search**:
```python
# Find videos matching description
query = "person doing high intensity interval training"
query_embedding = clip.encode_text(query)
# Compare with all video thumbnail embeddings
```

### 2.4 Why CLIP for Our Project?

**YouTube Thumbnails Analysis**:
- Understand visual branding (colors, layouts, aesthetics)
- Identify product placements in thumbnails
- Cluster videos by visual similarity
- Detect brand-specific visual patterns

**Advantages**:
- Pre-trained on diverse internet images (good generalization)
- Understands fitness/product concepts without fine-tuning
- Fixed 512-dim output (efficient for large-scale processing)
- Can link visual and textual brand signals

In [ ]:
# Example: CLIP Usage (pseudocode for learning)
import torch
import clip
from PIL import Image

# Load pre-trained CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# Process an image
image = preprocess(Image.open("youtube_thumbnail.jpg")).unsqueeze(0).to(device)

# Extract image embedding
with torch.no_grad():
    image_embedding = model.encode_image(image)
    image_embedding /= image_embedding.norm(dim=-1, keepdim=True)  # Normalize

print(f"Image embedding shape: {image_embedding.shape}")  # [1, 512]

# Extract text embedding
text = clip.tokenize(["fitness workout with dumbbells"]).to(device)
with torch.no_grad():
    text_embedding = model.encode_text(text)
    text_embedding /= text_embedding.norm(dim=-1, keepdim=True)

# Calculate similarity
similarity = (image_embedding @ text_embedding.T).item()
print(f"Image-text similarity: {similarity:.3f}")

## 3. BERT: Bidirectional Encoder Representations from Transformers

### 3.1 What is BERT?

**Developed by**: Google (2018)

**Core Idea**: Pre-train a bidirectional transformer on large text corpus to understand language context, then fine-tune for specific tasks.

**Key Innovation**: Bidirectional context
- Reads text left-to-right AND right-to-left simultaneously
- Captures full context around each word
- Pre-trained on masked language modeling and next sentence prediction

### 3.2 BERT Architecture

```
Input Text: "This fitness influencer promotes protein powder"
           ↓
Tokenization: [CLS] this fitness influencer promotes protein powder [SEP]
           ↓
Transformer Layers (12 or 24 layers)
- Self-attention mechanisms
- Bidirectional context flow
           ↓
Outputs:
- Token embeddings (768-dim per token)
- [CLS] token embedding (768-dim) → Sentence/document representation
```

**Variants**:
- **BERT-base**: 12 layers, 768-dim embeddings, 110M parameters
- **BERT-large**: 24 layers, 1024-dim embeddings, 340M parameters
- **DistilBERT**: Smaller, faster (66M params), 97% performance
- **RoBERTa**: Optimized BERT training

### 3.3 BERT Capabilities

**Text Classification**:
```python
# Classify text into categories
text = "This video is sponsored by Gymshark"
embedding = bert.encode(text)  # [CLS] token embedding
category = classifier(embedding)  # "sponsored content"
```

**Semantic Similarity**:
```python
# Find similar video descriptions
desc1 = bert.encode("Full body HIIT workout at home")
desc2 = bert.encode("High intensity home training session")
similarity = cosine_similarity(desc1, desc2)  # High similarity
```

**Clustering**:
```python
# Group similar content
embeddings = [bert.encode(text) for text in all_descriptions]
clusters = kmeans.fit_predict(embeddings)
# Cluster 0: Workout videos, Cluster 1: Product reviews, etc.
```

### 3.4 Why BERT for Our Project?

**Text Understanding**:
- Video titles and descriptions
- YouTube comments
- Tweet content
- Reddit posts

**Applications**:
- Identify brand mentions in context
- Understand sponsorship language
- Cluster content by topic
- Extract semantic features for brand-influencer matching

**Advantages**:
- Understands context and nuance
- Pre-trained on diverse text (good generalization)
- Captures brand-related semantic patterns
- Fixed 768-dim output (consistent features)

In [ ]:
# Example: BERT Usage (pseudocode for learning)
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

# Process text
text = "10 minute morning yoga routine for beginners"
inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=512)

# Extract embeddings
with torch.no_grad():
    outputs = model(**inputs)
    # [CLS] token embedding (sentence representation)
    sentence_embedding = outputs.last_hidden_state[:, 0, :]
    # Or use pooled output
    pooled_embedding = outputs.pooler_output

print(f"Sentence embedding shape: {sentence_embedding.shape}")  # [1, 768]
print(f"Pooled embedding shape: {pooled_embedding.shape}")  # [1, 768]

## 4. Why CLIP and BERT for Influencer-Brand Mapping?

### 4.1 Multi-Modal Data

**Visual Signals (CLIP)**:
- YouTube thumbnail aesthetics → Brand visual identity
- Product placement visibility → Brand partnerships
- Video style and quality → Influencer tier

**Textual Signals (BERT)**:
- Video descriptions → Explicit brand mentions
- Comments → Audience brand associations
- Tweets → Sponsorship announcements
- Reddit posts → Authentic brand discussions

### 4.2 Complementary Information

**CLIP captures**:
- Visual branding (logos, colors, packaging)
- Product categories (apparel, supplements, equipment)
- Video production quality

**BERT captures**:
- Explicit brand names and mentions
- Sentiment and tone toward brands
- Sponsorship language patterns
- Topical relevance

**Together**:
- Comprehensive influencer-brand relationship understanding
- Cross-validation (visual + textual brand signals)
- Richer features for downstream tasks (GAIL, matching, etc.)

### 4.3 Transfer Learning Benefits

**Pre-trained Models**:
- CLIP: Trained on 400M image-text pairs
- BERT: Trained on BooksCorpus + Wikipedia (3.3B words)

**Our Benefits**:
- No need for large labeled dataset
- Good performance out-of-the-box
- Can fine-tune if needed with small dataset
- State-of-the-art feature quality

### 4.4 Scalability

**Fixed-size Embeddings**:
- CLIP: 512 dimensions
- BERT: 768 dimensions
- Total: 1,280 dimensions per content item

**Efficient Processing**:
```python
# Process 1M videos:
# - Visual: 1M × 512 × 4 bytes = 2 GB
# - Textual: 1M × 768 × 4 bytes = 3 GB
# - Total: ~5 GB (manageable)
```

**Fast Similarity Search**:
- Use libraries like FAISS for efficient nearest neighbor search
- Enable real-time influencer-brand matching

## 5. Feature Extraction Pipeline

### 5.1 Visual Feature Extraction with CLIP

**Pipeline**:
```
1. Download YouTube thumbnail URLs
2. Preprocess images (resize, normalize)
3. Batch processing through CLIP image encoder
4. Extract 512-dim embeddings
5. Save to disk (numpy/pickle)
```

**Preprocessing**:
```python
from torchvision import transforms

clip_preprocess = transforms.Compose([
    transforms.Resize(224),           # CLIP expects 224×224
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],  # CLIP normalization
        std=[0.26862954, 0.26130258, 0.27577711]
    )
])
```

**Batch Processing** (for efficiency):
```python
batch_size = 256  # Adjust based on GPU memory
for i in range(0, len(images), batch_size):
    batch = images[i:i+batch_size]
    with torch.no_grad():
        embeddings = clip_model.encode_image(batch)
    # Save embeddings
```

### 5.2 Textual Feature Extraction with BERT

**Pipeline**:
```
1. Load text data (titles, descriptions, comments)
2. Preprocess text (clean, truncate)
3. Tokenize with BERT tokenizer
4. Batch processing through BERT encoder
5. Extract 768-dim embeddings ([CLS] token)
6. Save to disk
```

**Text Preprocessing**:
```python
def preprocess_text(text):
    # Clean text
    text = text.lower()
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'\s+', ' ', text)     # Normalize whitespace
    # Truncate to BERT max length (512 tokens)
    return text[:2000]  # Approximate token limit
```

**Handling Long Texts**:
```python
# Option 1: Truncate
inputs = tokenizer(text, max_length=512, truncation=True)

# Option 2: Chunk and average
chunks = [text[i:i+2000] for i in range(0, len(text), 2000)]
embeddings = [bert_encode(chunk) for chunk in chunks]
final_embedding = np.mean(embeddings, axis=0)
```

### 5.3 Multi-Modal Feature Integration

**Concatenation**:
```python
# Simple concatenation
visual_emb = clip_encode(thumbnail)      # 512-dim
textual_emb = bert_encode(description)   # 768-dim
combined = np.concatenate([visual_emb, textual_emb])  # 1280-dim
```

**Weighted Combination**:
```python
# Weight by modality importance
alpha = 0.4  # Visual weight
beta = 0.6   # Textual weight
combined = alpha * visual_emb + beta * textual_emb
# Note: Need same dimensionality, so project to common space first
```

**Learned Fusion** (advanced):
```python
# Neural network to learn optimal combination
fusion_layer = nn.Linear(512 + 768, 512)
combined = fusion_layer(torch.cat([visual_emb, textual_emb]))
```

## 6. Pre-trained vs. Fine-tuned Models

### 6.1 Pre-trained Models (Default Approach)

**Advantages**:
- ✅ Ready to use immediately
- ✅ No training data required
- ✅ Good general performance
- ✅ Fast deployment
- ✅ Lower computational cost

**Disadvantages**:
- ❌ May miss domain-specific patterns
- ❌ Generic embeddings (not optimized for our task)

**When to Use**:
- Starting point for any project
- When lacking labeled training data
- When task is similar to pre-training (image classification, text understanding)

### 6.2 Fine-tuned Models

**What is Fine-tuning?**:
- Continue training pre-trained model on domain-specific data
- Adjust weights to capture task-specific patterns
- Typically requires labeled dataset

**Advantages**:
- ✅ Better performance on specific domain
- ✅ Captures influencer/fitness/brand-specific patterns
- ✅ Can optimize for specific task (e.g., brand matching)

**Disadvantages**:
- ❌ Requires labeled data (influencer-brand pairs)
- ❌ Computational cost (GPU time)
- ❌ Risk of overfitting on small dataset
- ❌ Longer development time

**When to Use**:
- After evaluating pre-trained model performance
- When domain is very specific (e.g., medical, legal)
- When labeled dataset is available
- When performance gains justify cost

### 6.3 Decision Framework

**Step 1**: Start with pre-trained models
```python
# Use CLIP and BERT out-of-the-box
clip_embeddings = extract_clip_features(thumbnails)
bert_embeddings = extract_bert_features(descriptions)
```

**Step 2**: Evaluate on sample task
```python
# Example: Brand-influencer similarity
# Can we find Nike-sponsored influencers by similarity to Nike brand profile?
precision_at_10 = evaluate_retrieval(embeddings, ground_truth)
```

**Step 3**: Decide based on results
```python
if precision_at_10 > 0.7:
    decision = "Pre-trained sufficient"
elif precision_at_10 > 0.5:
    decision = "Consider fine-tuning"
else:
    decision = "Fine-tuning recommended"
```

### 6.4 Fine-tuning Approaches (If Needed)

**Approach 1: Contrastive Learning**
```python
# Learn to bring influencer and their brands closer
# Maximize similarity for (influencer, sponsored_brand) pairs
# Minimize similarity for (influencer, unrelated_brand) pairs
```

**Approach 2: Classification**
```python
# Train classifier: Given influencer features, predict brand category
# Use embeddings as input to classifier
# Fine-tune embeddings through backpropagation
```

**Approach 3: Metric Learning**
```python
# Learn distance metric optimized for brand matching
# Triplet loss: (anchor, positive, negative)
```

## 7. Research Paper - Methodology Section

### Feature Extraction Methodology

**Abstract**:
"We employ state-of-the-art multi-modal feature extraction using CLIP for visual embeddings and BERT for textual embeddings. This approach captures both visual and semantic brand signals from social media content, enabling comprehensive influencer-brand relationship modeling."

### 7.1 Visual Feature Extraction

**Model**: CLIP (ViT-B/32)
- **Architecture**: Vision Transformer with 32×32 patches
- **Embedding Dimension**: 512
- **Pre-training**: 400M image-text pairs from internet

**Input**: YouTube video thumbnails (1920×1080 → 224×224)

**Preprocessing**:
1. Resize to 224×224 pixels
2. Center crop
3. Normalize with CLIP statistics

**Extraction Process**:
1. Download thumbnails via YouTube API URLs
2. Batch process (256 images/batch) through CLIP image encoder
3. L2-normalize embeddings for cosine similarity
4. Store as numpy arrays (memory-efficient)

**Justification**: CLIP's joint image-text training enables understanding of visual brand elements (logos, products, aesthetics) without domain-specific fine-tuning.

### 7.2 Textual Feature Extraction

**Model**: BERT-base-uncased
- **Architecture**: 12-layer Transformer
- **Embedding Dimension**: 768
- **Pre-training**: BooksCorpus + English Wikipedia

**Input Sources**:
- YouTube: Video titles, descriptions, top comments
- Twitter: Tweet text, user bios
- Reddit: Post titles and content

**Preprocessing**:
1. Lowercase conversion
2. URL removal
3. Truncation to 512 tokens
4. Special character normalization

**Extraction Process**:
1. Tokenize with BERT tokenizer
2. Batch process (128 texts/batch) through BERT encoder
3. Extract [CLS] token embedding (sentence representation)
4. L2-normalize for consistency
5. Store as numpy arrays

**Justification**: BERT's bidirectional context understanding captures brand mentions, sponsorship language, and semantic content relationships.

### 7.3 Multi-Modal Integration

**Concatenation Strategy**:
- Visual (512-dim) + Textual (768-dim) = 1,280-dim combined embedding
- Preserves full information from both modalities

**Applications**:
1. **Similarity Search**: Find influencers with similar multi-modal profiles
2. **Clustering**: Group content by combined visual/textual patterns
3. **Brand Matching**: Align influencer embeddings with brand embeddings
4. **Trend Analysis**: Temporal evolution of visual/textual brand signals

### 7.4 Evaluation Plan

**Intrinsic Evaluation**:
- Embedding space visualization (t-SNE, UMAP)
- Cluster coherence metrics
- Inter vs. intra-cluster distances

**Extrinsic Evaluation**:
- Brand-influencer retrieval accuracy
- Sponsorship prediction from embeddings
- Comparison with baseline features (TF-IDF, ResNet)

**Fine-tuning Decision**:
- Evaluate pre-trained performance on sample tasks
- If precision@10 < 0.7, proceed with fine-tuning
- Document improvements and computational costs

### 7.5 Expected Results

**Embedding Quality**:
- Meaningful visual clusters (e.g., yoga vs. weightlifting)
- Semantic textual groupings (e.g., nutrition vs. training)
- Cross-modal alignment (matching visual/textual brand signals)

**Performance Metrics**:
- Brand retrieval precision@10: > 0.7
- Embedding space separability (silhouette score): > 0.4
- Cross-modal similarity correlation: > 0.6

**Dataset Scale**:
- ~2M video visual embeddings
- ~5M text embeddings (videos + comments + tweets)
- Storage: ~5-8 GB total
- Processing time: ~8-12 hours on GPU

## 8. Implementation Checklist

### Phase 1: CLIP Visual Features
- [ ] Install CLIP library
- [ ] Download and test CLIP model
- [ ] Implement thumbnail download pipeline
- [ ] Create batch processing function
- [ ] Extract embeddings for all thumbnails
- [ ] Save embeddings efficiently
- [ ] Visualize embedding space (t-SNE)

### Phase 2: BERT Textual Features
- [ ] Install transformers library
- [ ] Load BERT model
- [ ] Implement text preprocessing
- [ ] Create batch processing function
- [ ] Extract embeddings for titles/descriptions
- [ ] Extract embeddings for comments/tweets
- [ ] Save embeddings efficiently
- [ ] Visualize embedding space

### Phase 3: Integration
- [ ] Combine visual + textual embeddings
- [ ] Test similarity search
- [ ] Create embedding index (FAISS)
- [ ] Evaluate on sample tasks
- [ ] Document results

### Phase 4: Fine-tuning (Conditional)
- [ ] Prepare training pairs
- [ ] Implement fine-tuning pipeline
- [ ] Train and validate
- [ ] Compare with pre-trained
- [ ] Document improvements

## 9. Key Takeaways

1. **CLIP**: Captures visual brand signals from thumbnails (512-dim)
2. **BERT**: Captures textual brand mentions and semantics (768-dim)
3. **Multi-modal**: Combined features provide comprehensive understanding
4. **Pre-trained first**: Start with off-the-shelf models, evaluate, then decide on fine-tuning
5. **Scalable**: Fixed-size embeddings enable efficient processing of millions of items

## References

**CLIP**:
- Radford et al. (2021): "Learning Transferable Visual Models From Natural Language Supervision"
- OpenAI CLIP: https://github.com/openai/CLIP

**BERT**:
- Devlin et al. (2019): "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding"
- HuggingFace Transformers: https://huggingface.co/transformers/

## Next Steps

Proceed to implementation notebooks:
1. CLIP visual feature extraction
2. BERT textual feature extraction
3. Multi-modal feature integration and analysis

In [ ]:
print("\n" + "="*60)
print("CLIP and BERT Research Complete!")
print("="*60)
print("\nKey Concepts Covered:")
print("✓ Embedding fundamentals")
print("✓ CLIP architecture and applications")
print("✓ BERT architecture and applications")
print("✓ Multi-modal feature extraction pipeline")
print("✓ Pre-trained vs. fine-tuned models")
print("✓ Research methodology for paper")
print("\nReady to implement feature extraction!")